# Parte 2 — Experimentos no Wiki-RfA

Responsável por BFS e Dijkstra: Brenda. Revisor: André.

Este notebook lê o CSV tratado produzido por baixar_wikiRfA.ipynb. Execute as células em ordem, em uma sessão nova. Não é necessário executar o notebook de tratamento novamente se o CSV já estiver disponível.

**Pergunta de pesquisa:** Como a transformação dos pesos negativos para permitir o uso de Dijkstra altera os caminhos mínimos e a interpretação dos custos no grafo Wiki-RfA?

Esta etapa contém apenas os experimentos de BFS, que encontram menores distâncias em quantidade de arcos, ignorando os pesos. Transformação, Dijkstra e benchmark serão tratados em etapas posteriores.

**Apoio de IA:** ChatGPT (Codex) auxiliou na organização deste notebook e no código de chamada e exportação. O BFS utilizado é a implementação existente do projeto. As células foram criadas sem execução.


## 1. Preparação

Localiza a raiz procurando as pastas src e data no diretório atual e em seus ancestrais. Permite iniciar o notebook pela raiz ou pela pasta do dataset, sem depender das variáveis de outro notebook.


In [10]:
# [IA] ChatGPT (Codex) – localizar o projeto e importar as funções existentes
import json
import math
import sys
from pathlib import Path

raiz_projeto = Path.cwd().resolve()
while not ((raiz_projeto / "src").is_dir() and (raiz_projeto / "data").is_dir()):
    if raiz_projeto == raiz_projeto.parent:
        raise FileNotFoundError("Abra o notebook dentro da pasta do projeto.")
    raiz_projeto = raiz_projeto.parent

if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

from src.graphs.io import carregar_dataset
from src.graphs.algorithms import bfs

print("Raiz do projeto:", raiz_projeto)


Raiz do projeto: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1


## 2. Carregamento do grafo

Lê o CSV tratado com a coluna PESO. Preserva direção, laços e arcos paralelos, sem reaplicar o incremento nem alterar o arquivo. BFS ignora os valores dos pesos.


In [11]:
# [IA] ChatGPT (Codex) – carregar o CSV tratado para os experimentos
arquivo_tratado = raiz_projeto / "data" / "dataset_parte2" / "processed" / "wikiRfA_tratado.csv"
grafo_dataset = carregar_dataset(arquivo_tratado, coluna_peso="PESO")
print("Vértices:", len(grafo_dataset))


Vértices: 11376


## 3. BFS em três fontes

Fontes selecionadas a partir dos graus de saída salvos nas métricas: Siva1979 (grau 1.232), 'sed (grau 3, próximo da mediana dos graus positivos) e !---slappdash---! (grau 1). Empates foram resolvidos pelo nome, em ordem lexicográfica do Python, mantendo os identificadores originais.

A quantidade de outros usuários alcançados exclui a própria fonte. A maior distância considera apenas distâncias finitas, incluindo a distância zero da fonte. Para usuários inalcançáveis, a distância é registrada como null no JSON. Predecessor null pode indicar a fonte ou um usuário inalcançável; consulte também a distância.



In [12]:
# [IA] ChatGPT (Codex) – chamar o BFS existente e organizar seus resultados
fontes = ["Siva1979", "'sed", "!---slappdash---!"]
for fonte in fontes:
    if fonte not in grafo_dataset:
        raise ValueError(f"Fonte não encontrada no grafo: {fonte}")

resultados_bfs = []
for fonte in fontes:
    distancias, predecessores = bfs(grafo_dataset, fonte)
    distancias_json = {}
    outros_alcancados = 0
    maior_distancia = 0

    for usuario, distancia in distancias.items():
        if distancia == math.inf:
            distancias_json[usuario] = None
        else:
            distancias_json[usuario] = distancia
            if usuario != fonte:
                outros_alcancados += 1
            if distancia > maior_distancia:
                maior_distancia = distancia

    resultados_bfs.append({
        "fonte": fonte,
        "grau_saida_fonte": len(grafo_dataset[fonte]),
        "outros_usuarios_alcancados": outros_alcancados,
        "maior_distancia_finita": maior_distancia,
        "distancias": distancias_json,
        "predecessores": predecessores,
    })
    print(f"{fonte}: {outros_alcancados} outros usuários alcançados; maior distância = {maior_distancia} arcos.")


Siva1979: 3443 outros usuários alcançados; maior distância = 5 arcos.
'sed: 3444 outros usuários alcançados; maior distância = 6 arcos.
!---slappdash---!: 3444 outros usuários alcançados; maior distância = 7 arcos.


## 4. Exportação dos resultados

Salva os resultados em out/parte2/bfs_fontes.json, identificando a entrada e a unidade das distâncias. Executar novamente sobrescreve apenas esse arquivo. Não altera o CSV ou o JSON das métricas. Este arquivo não substitui o relatório de benchmark parte2_report.json.


In [13]:
# [IA] ChatGPT (Codex) – exportar os resultados das três fontes em JSON
relatorio_bfs = {
    "algoritmo": "BFS",
    "arquivo_entrada": arquivo_tratado.relative_to(raiz_projeto).as_posix(),
    "coluna_peso_carregada": "PESO",
    "pesos_considerados_na_busca": False,
    "unidade_distancia": "arcos",
    "vertices": len(grafo_dataset),
    "resultados": resultados_bfs,
}
pasta_saida = raiz_projeto / "out" / "parte2"
pasta_saida.mkdir(parents=True, exist_ok=True)
arquivo_bfs = pasta_saida / "bfs_fontes.json"
with open(arquivo_bfs, "w", encoding="utf-8") as arquivo:
    json.dump(relatorio_bfs, arquivo, ensure_ascii=False, indent=2, allow_nan=False)

print("Resultados salvos em:", arquivo_bfs)


Resultados salvos em: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1\out\parte2\bfs_fontes.json


## 5. Transformação dos pesos para Dijkstra

Transformação aprovada por Brenda: novo peso = PESO + 1. O menor PESO conferido no CSV foi -1; por isso, o deslocamento de 1 torna os pesos não negativos. A versão transformada fica em grafo_dijkstra, sem modificar grafo_dataset nem o CSV. Direção, ordem dos arcos, laços e arcos paralelos são preservados.

Para um caminho com k arcos, o custo transformado é o custo em PESO mais k. Portanto, esta transformação pode mudar os caminhos mínimos e não deve ser apresentada como preservação dos custos ou dos caminhos originais. PESO já inclui o incremento por repetição e não é a coluna VOTE original.

Esta célula apenas prepara e confere o grafo transformado. Não executa Dijkstra nem salva arquivos.


In [14]:
# [IA] ChatGPT (Codex) – aplicar o deslocamento de pesos aprovado por Brenda em um grafo separado
deslocamento_peso = 1
grafo_dijkstra = {}
arcos_originais = 0
arcos_transformados = 0

for origem, vizinhos in grafo_dataset.items():
    grafo_dijkstra[origem] = []
    arcos_originais += len(vizinhos)
    for destino, peso in vizinhos:
        novo_peso = peso + deslocamento_peso
        assert novo_peso >= 0, "O deslocamento não eliminou todos os pesos negativos."
        grafo_dijkstra[origem].append((destino, novo_peso))
        arcos_transformados += 1

assert len(grafo_dijkstra) == len(grafo_dataset)
assert arcos_transformados == arcos_originais
print("Transformação: novo peso = PESO +", deslocamento_peso)
print("Vértices preservados:", len(grafo_dijkstra))
print("Arcos preservados:", arcos_transformados)
print("Todos os pesos transformados são não negativos.")


Transformação: novo peso = PESO + 1
Vértices preservados: 11376
Arcos preservados: 196614
Todos os pesos transformados são não negativos.


## 6. Dijkstra em cinco pares

Executar após a transformação da seção 5. Os pares foram selecionados a partir dos resultados de BFS, usando as três fontes e destinos nos níveis 1, 5, 2, 6 e 7, respectivamente. Para cada nível escolhido, foi usado o primeiro destino em ordem lexicográfica do Python.

O Dijkstra existente será executado uma vez por origem; os resultados serão consultados para os cinco pares. Os caminhos minimizam a soma de PESO + 1 e podem diferir dos caminhos BFS. O custo registrado refere-se exclusivamente aos pesos transformados, não ao custo mínimo em PESO nem aos votos originais.

Os caminhos registram a sequência de vértices. Havendo arcos paralelos, essa sequência não identifica individualmente cada ocorrência; o custo transformado vem das distâncias devolvidas pelo algoritmo. Não há medição de tempo nesta etapa.


In [15]:
# [IA] ChatGPT (Codex) – chamar Dijkstra e a reconstrução existentes para os cinco pares aprovados
from src.graphs.algorithms import dijkstra, reconstruir_caminho

pares_dijkstra = [
    ("Siva1979", "(aeropagitica)"),
    ("Siva1979", "Dieter Simon"),
    ("'sed", "9cds"),
    ("'sed", "Caltrop"),
    ("!---slappdash---!", "Caltrop"),
]

for origem, destino in pares_dijkstra:
    if origem not in grafo_dijkstra or destino not in grafo_dijkstra:
        raise ValueError(f"Origem ou destino fora do grafo: {origem} -> {destino}")

buscas_dijkstra = {}
resultados_dijkstra = []
for origem, destino in pares_dijkstra:
    if origem not in buscas_dijkstra:
        buscas_dijkstra[origem] = dijkstra(grafo_dijkstra, origem)
    distancias, predecessores = buscas_dijkstra[origem]

    if distancias[destino] == math.inf:
        status = "SEM_CAMINHO"
        caminho = []
        custo = None
        quantidade_arcos = None
    else:
        status = "OK"
        caminho = reconstruir_caminho(distancias, predecessores, destino)
        custo = distancias[destino]
        quantidade_arcos = len(caminho) - 1

    resultados_dijkstra.append({
        "origem": origem,
        "destino": destino,
        "status": status,
        "caminho": caminho,
        "quantidade_arcos": quantidade_arcos,
        "custo_transformado": custo,
    })
    print(f"{origem} -> {destino}: {status}; custo transformado = {custo}; arcos = {quantidade_arcos}")


Siva1979 -> (aeropagitica): OK; custo transformado = 0; arcos = 3
Siva1979 -> Dieter Simon: OK; custo transformado = 6; arcos = 5
'sed -> 9cds: OK; custo transformado = 2; arcos = 2
'sed -> Caltrop: OK; custo transformado = 8; arcos = 7
!---slappdash---! -> Caltrop: OK; custo transformado = 8; arcos = 8


## 7. Exportação dos cinco pares

Salva out/parte2/dijkstra_pares.json, identificando a transformação e os resultados dos cinco pares. Executar novamente sobrescreve esse arquivo. Não modifica o CSV, os grafos, as métricas ou o resultado BFS. Não substitui o relatório de benchmark.


In [16]:
# [IA] ChatGPT (Codex) – exportar caminhos e custos transformados dos cinco pares
relatorio_dijkstra = {
    "algoritmo": "DIJKSTRA",
    "arquivo_entrada": arquivo_tratado.relative_to(raiz_projeto).as_posix(),
    "coluna_peso_base": "PESO",
    "transformacao": "novo_peso = PESO + deslocamento",
    "deslocamento": deslocamento_peso,
    "vertices": len(grafo_dijkstra),
    "buscas_por_origem": len(buscas_dijkstra),
    "resultados": resultados_dijkstra,
}
pasta_saida = raiz_projeto / "out" / "parte2"
pasta_saida.mkdir(parents=True, exist_ok=True)
arquivo_dijkstra = pasta_saida / "dijkstra_pares.json"
with open(arquivo_dijkstra, "w", encoding="utf-8") as arquivo:
    json.dump(relatorio_dijkstra, arquivo, ensure_ascii=False, indent=2, allow_nan=False)

print("Resultados salvos em:", arquivo_dijkstra)


Resultados salvos em: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1\out\parte2\dijkstra_pares.json
